# Train Influence Predictor (LOW / MEDIUM / HIGH)

**Luồng:** import → helper đánh giá → cấu hình → load dữ liệu → định nghĩa mô hình → chọn trên val → fit train+val → đánh giá test → lưu file.

| Phần | Nội dung |
|------|----------|
| 1–3 | Thư viện, `EvalResult` / `_evaluate`, `print_full_evaluation` |
| 4–5 | Đường dẫn, đọc CSV + label encoder |
| 6–9 | Ứng viên, kiểm tra model cũ, huấn luyện & chọn mô hình, đánh giá val |
| 10–13 | Fit train+val, test, lưu CSV / joblib / metadata |


## 1. Import thư viện


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 2. `EvalResult`, `_evaluate`, `_rank_key`


In [ ]:
@dataclass(frozen=True)
class EvalResult:
    macro_f1: float
    weighted_f1: float
    accuracy: float
    mae_class_distance: float

    def to_dict(self) -> dict[str, Any]:
        return {
            "macro_f1": self.macro_f1,
            "weighted_f1": self.weighted_f1,
            "accuracy": self.accuracy,
            "mae_class_distance": self.mae_class_distance,
        }


def _evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> EvalResult:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    accuracy = float(accuracy_score(y_true, y_pred))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return EvalResult(
        macro_f1=macro_f1,
        weighted_f1=weighted_f1,
        accuracy=accuracy,
        mae_class_distance=mae,
    )


def _rank_key(res: EvalResult) -> tuple[float, float]:
    return (res.macro_f1, -res.mae_class_distance)


## 3. Đánh giá đầy đủ (Accuracy, Precision, Recall, F1, report, CM)


In [ ]:
def print_full_evaluation(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_order: list[int],
    int_to_label: dict[str, str],
    title: str = "Evaluation",
) -> np.ndarray:
    labels = np.array(sorted(label_order))
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    names = [int_to_label[str(i)] for i in labels]

    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    prec_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    rec_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  {title}")
    print(sep)
    print(f"Accuracy:           {acc:.4f}")
    print(f"Precision (macro):  {prec_macro:.4f}   (weighted): {prec_weighted:.4f}")
    print(f"Recall (macro):     {rec_macro:.4f}   (weighted): {rec_weighted:.4f}")
    print(f"F1 (macro):         {f1_macro:.4f}   (weighted): {f1_weighted:.4f}")
    print("\nClassification report (per class):")
    print(
        classification_report(
            y_true, y_pred, labels=labels, target_names=names, digits=4, zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print("Confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(cm, index=[f"true_{n}" for n in names], columns=[f"pred_{n}" for n in names]))

    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=names)
    disp.plot(ax=ax, cmap="Blues", colorbar=True)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return cm


## 4. Cấu hình đường dẫn


In [ ]:
from pathlib import Path

BASE_DIR = Path(".").resolve()
if not (BASE_DIR / "feature_engineering_output").exists() and (
    BASE_DIR.parent / "feature_engineering_output"
).exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "feature_engineering_output"
MODEL_OUT_DIR = BASE_DIR / "model" / "influence_predictor"
FORCE_RETRAIN = True

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("MODEL_OUT_DIR:", MODEL_OUT_DIR)


## 5. Đọc `label_encoder` và các tập CSV


In [ ]:
label_encoder_path = DATA_DIR / "label_encoder.json"
label_encoder = json.loads(label_encoder_path.read_text(encoding="utf-8"))
label_to_int: dict[str, int] = {k: int(v) for k, v in label_encoder["label_to_int"].items()}
int_to_label: dict[str, str] = label_encoder["int_to_label"]
label_order = sorted(label_to_int.values())

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train_df = pd.read_csv(DATA_DIR / "y_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val_df = pd.read_csv(DATA_DIR / "y_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test_df = pd.read_csv(DATA_DIR / "y_test.csv")

feature_names = list(X_train.columns)
for name, x in [("X_val", X_val), ("X_test", X_test)]:
    if list(x.columns) != feature_names:
        raise ValueError(f"{name} columns differ from X_train.")

y_train = y_train_df["label_int"].astype(int).to_numpy()
y_val = y_val_df["label_int"].astype(int).to_numpy()
y_test = y_test_df["label_int"].astype(int).to_numpy()
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)


## 6. Danh sách mô hình ứng viên


In [ ]:
candidates: list[tuple[str, Any]] = [
    (
        "logreg_standard_scaler",
        Pipeline(
            steps=[
                ("scaler", StandardScaler(with_mean=True, with_std=True)),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=3000,
                        solver="lbfgs",
                        class_weight="balanced",
                        n_jobs=None,
                        random_state=42,
                    ),
                ),
            ]
        ),
    ),
    (
        "rf_balanced_subsample",
        RandomForestClassifier(
            n_estimators=600,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample",
            min_samples_leaf=1,
        ),
    ),
    (
        "extratrees_balanced",
        ExtraTreesClassifier(
            n_estimators=900,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced",
            min_samples_leaf=1,
        ),
    ),
    (
        "histgb",
        HistGradientBoostingClassifier(
            max_depth=6,
            learning_rate=0.05,
            max_iter=500,
            random_state=42,
        ),
    ),
]


## 7. Đường dẫn file output & kiểm tra model đã tồn tại


In [ ]:
model_path = MODEL_OUT_DIR / "influence_predictor.joblib"
meta_path = MODEL_OUT_DIR / "metadata.json"
if model_path.exists() and meta_path.exists() and not FORCE_RETRAIN:
    raise SystemExit(f"Model đã tồn tại tại {MODEL_OUT_DIR}. Đặt FORCE_RETRAIN = True để huấn luyện lại.")


## 8. Huấn luyện từng ứng viên trên `X_train`, đo trên validation


In [ ]:
best_name = None
best_model = None
best_val_res = None

for name, model in candidates:
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    res = _evaluate(y_val, y_val_pred)
    print(
        f"[VAL] {name}: macro_f1={res.macro_f1:.4f}, weighted_f1={res.weighted_f1:.4f}, "
        f"acc={res.accuracy:.4f}, mae={res.mae_class_distance:.4f}"
    )
    if best_val_res is None or _rank_key(res) > _rank_key(best_val_res):
        best_name = name
        best_model = model
        best_val_res = res


## 9. Xác nhận mô hình được chọn & đánh giá chi tiết trên validation


In [ ]:
assert best_name is not None and best_model is not None and best_val_res is not None
print(f"\n>>> Chọn mô hình: {best_name}")
y_val_pred_best = best_model.predict(X_val)
print_full_evaluation(
    y_val, y_val_pred_best, label_order, int_to_label, title=f"Validation — {best_name}"
)


## 10. Huấn luyện lại trên train + validation


In [ ]:
X_train_all = pd.concat([X_train, X_val], axis=0, ignore_index=True)
y_train_all = np.concatenate([y_train, y_val], axis=0)
best_model.fit(X_train_all, y_train_all)


## 11. Dự đoán & đánh giá trên test


In [ ]:
y_test_pred = best_model.predict(X_test)
test_res = _evaluate(y_test, y_test_pred)
print_full_evaluation(
    y_test, y_test_pred, label_order, int_to_label, title=f"Test — {best_name}"
)


## 12. Lưu `test_predictions.csv`


In [ ]:
proba_path = MODEL_OUT_DIR / "test_predictions.csv"
pred_df = pd.DataFrame(
    {
        "y_true_int": y_test.astype(int),
        "y_true_label": [int_to_label[str(i)] for i in y_test.astype(int)],
        "y_pred_int": y_test_pred.astype(int),
        "y_pred_label": [int_to_label[str(i)] for i in y_test_pred.astype(int)],
    }
)
if hasattr(best_model, "predict_proba"):
    try:
        proba = best_model.predict_proba(X_test)
        pred_df["pred_proba_max"] = np.max(proba, axis=1)
    except Exception:
        pass
pred_df.to_csv(proba_path, index=False, encoding="utf-8")
print("Đã ghi:", proba_path)


## 13. Lưu model (`joblib`) và `metadata.json`


In [ ]:
joblib.dump(best_model, model_path)

# val_cm: dự đoán val từ mô hình đã chọn (chỉ fit trên X_train), khớp train_influence_predictor.py
val_cm = confusion_matrix(y_val, y_val_pred_best, labels=label_order)

timestamp = datetime.now(timezone.utc).isoformat()
metadata = {
    "timestamp_utc": timestamp,
    "data_dir": str(DATA_DIR),
    "feature_names": feature_names,
    "label_encoder": label_encoder,
    "selected_model": best_name,
    "val_metrics": best_val_res.to_dict(),
    "val_confusion_matrix": val_cm.tolist(),
    "test_metrics": test_res.to_dict(),
    "test_confusion_matrix": confusion_matrix(y_test, y_test_pred, labels=label_order).tolist(),
    "test_extended_metrics": {
        "precision_macro": float(
            precision_score(y_test, y_test_pred, average="macro", zero_division=0)
        ),
        "precision_weighted": float(
            precision_score(y_test, y_test_pred, average="weighted", zero_division=0)
        ),
        "recall_macro": float(recall_score(y_test, y_test_pred, average="macro", zero_division=0)),
        "recall_weighted": float(
            recall_score(y_test, y_test_pred, average="weighted", zero_division=0)
        ),
    },
    "splits": {
        "X_train": "X_train.csv",
        "y_train": "y_train.csv",
        "X_val": "X_val.csv",
        "y_val": "y_val.csv",
        "X_test": "X_test.csv",
        "y_test": "y_test.csv",
    },
}
meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved model: {model_path}")
print(f"Saved metadata: {meta_path}")
